# LECTURA DE LAS 3 TABLAS

In [25]:
import os
import re
import numpy as np
import pandas as pd
import pdfplumber
from tqdm import tqdm
from datetime import datetime

# ==================== CONFIGURACIÓN ====================
PDF_FOLDER = "../pdfs"
OUTPUT_CSV = "precios_huevo_consolidado.csv"
HISTORICO_CSV = "precios_huevo_consolidado.csv"  # mismo archivo para histórico

# Diccionario de meses
MESES = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'ago': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

# BBOX del cuadro izquierdo (ajústalo si es necesario)
BBOX_CUADRO1 = (30, 140, 395, 690)

# Patrones de expresiones regulares
PATRON_FECHA_REPORTE = re.compile(r'Fecha\s+(\d{1,2})/(\d{1,2})/(\d{4})', re.IGNORECASE)
PATRON_INICIO_FILA = re.compile(r'^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\b', re.IGNORECASE)
PATRON_NUMEROS = re.compile(r'\d+(?:\.\d+)?')
PATRON_BLOQUE_CUADRO = re.compile(
    r'PRECIOS\s+DE\s+HUEVO\s+SEGUN\s+TIPO\s+DE\s+COMERCIALIZACION.*?'
    r'Fecha\s+Mayorista\s+Minorista(.*?)'
    r'OFERTA\s+DEL\s+HUEVO\s+DE\s+PRIMERA\s+SEGUN\s+MACROREGION',
    re.IGNORECASE | re.DOTALL
)
PATRON_FALLBACK = re.compile(r'Fecha\s+Mayorista\s+Minorista(.*)', re.IGNORECASE | re.DOTALL)

print("✅ Configuración cargada")

✅ Configuración cargada


In [26]:
def normalizar_texto(texto):
    """Limpia espacios y saltos de línea"""
    if not texto:
        return ""
    lineas = [re.sub(r'\s+', ' ', ln).strip() for ln in texto.split("\n")]
    return "\n".join(lineas)

def extraer_texto_pagina(pdf_path, page_num=0):
    """Extrae texto de una página completa"""
    with pdfplumber.open(pdf_path) as pdf:
        if len(pdf.pages) <= page_num:
            return ""
        return normalizar_texto(pdf.pages[page_num].extract_text() or "")

def obtener_fecha_reporte_desde_texto(texto):
    """Obtiene día, mes, año del reporte desde la fecha en el PDF"""
    m = PATRON_FECHA_REPORTE.search(texto or "")
    return map(int, m.groups()) if m else (None, None, None)

def extraer_texto_bloque_izquierdo(pdf_path, bbox):
    """Extrae solo el área del cuadro izquierdo usando bbox"""
    with pdfplumber.open(pdf_path) as pdf:
        if len(pdf.pages) <= 0:
            return ""
        return normalizar_texto(pdf.pages[0].crop(bbox).extract_text() or "")

def aislar_texto_cuadro1(texto_bloque):
    """Aísla exclusivamente el contenido del cuadro de precios"""
    if not texto_bloque:
        return ""
    m = PATRON_BLOQUE_CUADRO.search(texto_bloque)
    if not m:
        m = PATRON_FALLBACK.search(texto_bloque)
    return normalizar_texto(m.group(1)) if m else ""

def parsear_filas_cuadro1(texto_cuadro, anio_reporte):
    """Convierte líneas del cuadro en filas del DataFrame"""
    filas = []
    lineas = [ln.strip() for ln in texto_cuadro.split("\n") if ln.strip()]
    
    for linea in lineas:
        linea_limpia = re.sub(r'\s+', ' ', linea.strip()).lower()
        
        # Filtros
        if any(x in linea_limpia for x in ["var%", "fuente", "a partir del", "no se realiza", "encuesta", "#n/a"]):
            continue
        
        m_inicio = PATRON_INICIO_FILA.match(linea_limpia)
        if not m_inicio:
            continue
        
        mes_txt, dia_txt = m_inicio.groups()
        mes_num = MESES.get(mes_txt.lower())
        if not mes_num:
            continue
        
        resto = PATRON_INICIO_FILA.sub("", linea_limpia, count=1).strip()
        numeros = PATRON_NUMEROS.findall(resto)
        
        if len(numeros) >= 2:
            mayorista, minorista = float(numeros[0]), float(numeros[1])
        elif len(numeros) == 1:
            mayorista, minorista = float(numeros[0]), np.nan
        else:
            continue
        
        filas.append({
            "anio": anio_reporte,
            "mes_txt": mes_txt.lower(),
            "mes_num": mes_num,
            "dia": int(dia_txt),
            "mayorista": mayorista,
            "minorista": minorista
        })
    
    return pd.DataFrame(filas)

def limpiar_cuadro1(df):
    """Limpia datos inválidos, duplicados y crea columna fecha"""
    if df.empty:
        return pd.DataFrame(columns=["fecha", "anio", "mes_txt", "mes_num", "dia", "mayorista", "minorista"])
    
    df = df.copy()
    df["fecha"] = pd.to_datetime(dict(year=df["anio"], month=df["mes_num"], day=df["dia"]), errors="coerce")
    df = df[df["fecha"].notna() & df["mayorista"].between(0.5, 20)]
    
    # Eliminar duplicados exactos
    df = df.drop_duplicates().sort_values("fecha").reset_index(drop=True)
    return df[["fecha", "anio", "mes_txt", "mes_num", "dia", "mayorista", "minorista"]]

def leer_cuadro1_pdf(pdf_path):
    """Pipeline completo para leer un PDF"""
    texto_pagina = extraer_texto_pagina(pdf_path)
    _, _, anio_reporte = obtener_fecha_reporte_desde_texto(texto_pagina)
    if not anio_reporte:
        raise ValueError(f"No se encontró fecha de reporte en {pdf_path}")
    
    texto_bloque = extraer_texto_bloque_izquierdo(pdf_path, BBOX_CUADRO1)
    texto_cuadro = aislar_texto_cuadro1(texto_bloque)
    df = parsear_filas_cuadro1(texto_cuadro, anio_reporte)
    return limpiar_cuadro1(df)

print("✅ Funciones auxiliares listas")

✅ Funciones auxiliares listas


In [27]:
# Cargar histórico si existe
if os.path.exists(OUTPUT_CSV):
    df_historico = pd.read_csv(OUTPUT_CSV, parse_dates=["fecha"])
    pdfs_procesados = set(df_historico["pdf_file"].unique()) if "pdf_file" in df_historico.columns else set()
else:
    df_historico = pd.DataFrame()
    pdfs_procesados = set()

# Listar todos los PDFs en la carpeta
todos_pdfs = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]

# Filtrar solo los nuevos
pdfs_nuevos = [f for f in todos_pdfs if f not in pdfs_procesados]

print(f"📁 Total PDFs en carpeta: {len(todos_pdfs)}")
print(f"✅ Ya procesados: {len(pdfs_procesados)}")
print(f"🆕 Nuevos por procesar: {len(pdfs_nuevos)}")
print("\nNuevos archivos:")
for p in pdfs_nuevos[:5]:
    print(f"  - {p}")

📁 Total PDFs en carpeta: 84
✅ Ya procesados: 0
🆕 Nuevos por procesar: 84

Nuevos archivos:
  - 09_abril_2026.pdf
  - 10_abril_2026.pdf
  - 12_diciembre_2025.pdf
  - 13_abril_2026.pdf
  - 13_febrero_2026.pdf


In [28]:
if pdfs_nuevos:
    lista_dfs = []
    lista_errores = []
    
    # Barra de progreso con porcentaje
    for file_name in tqdm(pdfs_nuevos, desc="📊 Procesando PDFs nuevos", unit="pdf", ncols=100):
        pdf_path = os.path.join(PDF_FOLDER, file_name)
        try:
            df_tmp = leer_cuadro1_pdf(pdf_path)
            df_tmp["pdf_file"] = file_name
            lista_dfs.append(df_tmp)
        except Exception as e:
            lista_errores.append({"pdf_file": file_name, "error": str(e)})
    
    df_nuevos = pd.concat(lista_dfs, ignore_index=True) if lista_dfs else pd.DataFrame()
    df_errores = pd.DataFrame(lista_errores)
    
    # Mostrar resumen
    print(f"\n✅ Procesados correctamente: {len(lista_dfs)} PDFs")
    print(f"❌ Errores: {len(lista_errores)} PDFs")
    print(f"📈 Filas extraídas: {len(df_nuevos)}")
    
    if not df_errores.empty:
        print("\n⚠️ Errores:")
        display(df_errores)
else:
    print("🎉 No hay PDFs nuevos para procesar")
    df_nuevos = pd.DataFrame()

📊 Procesando PDFs nuevos: 100%|███████████████████████████████████| 84/84 [03:37<00:00,  2.59s/pdf]


✅ Procesados correctamente: 84 PDFs
❌ Errores: 0 PDFs
📈 Filas extraídas: 1089


In [29]:
# Unir histórico con nuevos
if not df_nuevos.empty:
    if not df_historico.empty:
        df_total = pd.concat([df_historico, df_nuevos], ignore_index=True)
    else:
        df_total = df_nuevos
else:
    df_total = df_historico

# Eliminar duplicados exactos y por (fecha, pdf_file)
if not df_total.empty:
    df_total = df_total.drop_duplicates().reset_index(drop=True)
    if "pdf_file" in df_total.columns:
        df_total = df_total.drop_duplicates(subset=["fecha", "pdf_file"], keep="first")
    
    df_total = df_total.sort_values("fecha").reset_index(drop=True)
    
    # Guardar CSV final
    df_total.to_csv(OUTPUT_CSV, index=False)
    print(f"\n Archivo guardado: {OUTPUT_CSV}")
    print(f" Total registros finales: {len(df_total)}")
    print(f" Desde: {df_total['fecha'].min().date()} hasta: {df_total['fecha'].max().date()}")
    
    # Mostrar muestra
    display(df_total.head())
else:
    print(" No hay datos para guardar")


 Archivo guardado: precios_huevo_consolidado.csv
 Total registros finales: 1089
 Desde: 2025-09-03 hasta: 2026-04-22


,fecha,anio,mes_txt,mes_num,dia,mayorista,minorista,pdf_file
0,2025-09-03,2025,sep,9,3,5.70,7.15,16_setiembre_2025.pdf
1,2025-09-04,2025,sep,9,4,5.70,7.15,16_setiembre_2025.pdf
2,2025-09-04,2025,sep,9,4,5.70,7.15,17_setiembre_2025.pdf
3,2025-09-05,2025,sep,9,5,5.65,7.12,18_setiembre_2025.pdf
4,2025-09-05,2025,sep,9,5,5.65,7.12,17_setiembre_2025.pdf


In [30]:
if not df_total.empty:
    duplicados_fecha = df_total[df_total.duplicated(subset=["fecha"], keep=False)].sort_values("fecha")
    
    if not duplicados_fecha.empty:
        print(f"Hay {len(duplicados_fecha)} registros con fecha duplicada (misma fecha, distinto PDF):")
        display(duplicados_fecha)
    else:
        print(" No hay fechas duplicadas en el consolidado.")
else:
    print(" DataFrame vacío")

Hay 1068 registros con fecha duplicada (misma fecha, distinto PDF):


,fecha,anio,mes_txt,mes_num,dia,mayorista,minorista,pdf_file
1,2025-09-04,2025,sep,9,4,5.70,7.15,16_setiembre_2025.pdf
2,2025-09-04,2025,sep,9,4,5.70,7.15,17_setiembre_2025.pdf
3,2025-09-05,2025,sep,9,5,5.65,7.12,18_setiembre_2025.pdf
4,2025-09-05,2025,sep,9,5,5.65,7.12,17_setiembre_2025.pdf
5,2025-09-05,2025,sep,9,5,5.65,7.12,16_setiembre_2025.pdf
...,...,...,...,...,...,...,...,...
1082,2026-04-20,2026,abr,4,20,6.15,7.44,21_abril_2026.pdf
1084,2026-04-20,2026,abr,4,20,6.15,7.44,22_abril_2026.pdf
1086,2026-04-21,2026,abr,4,21,6.15,7.44,23_abril_2026.pdf
1085,2026-04-21,2026,abr,4,21,6.15,7.44,26_abril_2026.pdf


In [32]:
# los duplicados ocurren por el pdf_file la borramos porque no indica nada
df_total = df_total.drop(columns=["pdf_file"])

In [33]:
df_total.duplicated().sum()

848

In [36]:
df_total= df_total.drop_duplicates()

In [39]:
df_total.shape

(241, 7)

In [40]:
# descargar df_total como tabla_1.csv
df_total.to_csv("tabla_1.csv", index=False)